# Table 4: within-participant environmental contrasts
Original numerical analysis: participant-specific median splits, at least 20 complete bouts per contrast, 5,000 bootstrap resamples (seed 7), and unadjusted Wilcoxon signed-rank tests. Each participant has equal weight.

**Reproduction caveats:** the source uses all non-TOPAL bouts with steps, not an explicit free-living time filter. Its “Clutter” row uses `EnvLowerCentralObstacleAreaRatioMean`, not the supplementary clutter-index formula. Preserve these choices to reproduce the reported numbers; see README before interpreting them.

In [2]:
# Table 4: Within-participant environmental exposure and gait variability.
from decimal import Decimal, ROUND_HALF_UP
from pathlib import Path

import numpy as np
import pandas as pd
from scipy.stats import wilcoxon
from IPython.display import display

BOUTS_PATH = Path('derived/bouts.parquet')
MIN_BOUTS = 20
N_BOOT = 5000
BOOTSTRAP_SEED = 7

OUTCOMES = {
    'STCoV': 'BoutStepTCV',
    'SaTCoV': 'BoutStanceTCV',
    'SwTCoV': 'BoutSwingTCV',
    'SrTCoV': 'BoutStrideTCV',
}
EXPOSURES = {
    'Open Floor High vs Low': 'EnvVisibleOpenFloorRatioMean',
    'Clutter High vs Low': 'EnvLowerCentralObstacleAreaRatioMean',
    'Corridor Occupancy High vs Low': 'EnvCentralCorridorOccupancyMean',
    'Path Obstruction High vs Low': 'EnvPathObstructionIndexMean',
    'Stairs High vs Low': 'EnvStairsPresentFraction',
}


In [3]:
# Preserve the source analysis: exclude TOPAL and bouts with no steps.
# Missing values are removed separately for each exposure/outcome pair below.
# The source's 3,096-bout complete-case diagnostic was not used to filter Table 4.
columns = ['Participant', 'BoutNumSteps', *OUTCOMES.values(), *EXPOSURES.values()]
bouts = pd.read_parquet(BOUTS_PATH, columns=columns)
bouts = bouts.loc[
    (bouts['BoutNumSteps'].fillna(0) > 0)
    & ~bouts['Participant'].str.startswith('TOPAL')
].copy()
bouts['Group'] = bouts['Participant'].map(
    lambda participant: 'PwP' if 'PD' in participant else 'Control'
)


In [4]:
def participant_high_low_deltas(df, exposure_col, outcome_col):
    """One high-minus-low mean difference per eligible participant."""
    complete = df[['Participant', exposure_col, outcome_col]].dropna()
    deltas = []
    for _, participant_bouts in complete.groupby('Participant'):
        if len(participant_bouts) < MIN_BOUTS:
            continue
        threshold = participant_bouts[exposure_col].median()
        low = participant_bouts.loc[
            participant_bouts[exposure_col] <= threshold, outcome_col
        ].mean()
        high = participant_bouts.loc[
            participant_bouts[exposure_col] > threshold, outcome_col
        ].mean()
        if not (np.isnan(low) or np.isnan(high)):
            deltas.append(float(high - low))
    return np.asarray(deltas, dtype=float)


def summarize_deltas(values):
    """Equal participant weighting; percentile bootstrap CI for the mean."""
    if len(values) == 0:
        raise ValueError('No participants have sufficient valid high/low bouts.')
    # Reset the RNG for each contrast, as in the source notebook.
    rng = np.random.default_rng(BOOTSTRAP_SEED)
    means = np.empty(N_BOOT)
    for i in range(N_BOOT):
        means[i] = rng.choice(values, size=len(values), replace=True).mean()
    ci_low, ci_high = np.quantile(means, [0.025, 0.975])
    return {
        'Mean Δ': float(values.mean()),
        'Median Δ': float(np.median(values)),
        '95% CI lower': float(ci_low),
        '95% CI upper': float(ci_high),
        'Fraction positive': float((values > 0).mean()),
        'Wilcoxon p': float(wilcoxon(values).pvalue),
    }


In [5]:
# Calculate all 40 contrasts in manuscript order, retaining full precision.
rows = []
for exposure_label, exposure_col in EXPOSURES.items():
    for outcome_label, outcome_col in OUTCOMES.items():
        for group in ['PwP', 'Control']:
            values = participant_high_low_deltas(
                bouts.loc[bouts['Group'] == group], exposure_col, outcome_col
            )
            rows.append({
                'Environmental factor': exposure_label,
                'Group': group,
                'Outcome': outcome_label,
                **summarize_deltas(values),
            })
table4 = pd.DataFrame(rows)


In [6]:
# Display only Table 4; round half up to three decimals, with p < 0.001.
def format_three_decimals(value):
    rounded = Decimal(str(value)).quantize(Decimal('0.001'), rounding=ROUND_HALF_UP)
    return '0.000' if rounded == 0 else str(rounded)


numeric_columns = [
    'Mean Δ', 'Median Δ', '95% CI lower', '95% CI upper', 'Fraction positive'
]
formatters = {column: format_three_decimals for column in numeric_columns}
formatters['Wilcoxon p'] = (
    lambda p: '<0.001' if p < 0.001 else format_three_decimals(p)
)
display(
    table4.set_index(['Environmental factor', 'Group', 'Outcome'])
    .style.format(formatters)
    .set_caption(
        'Table 4. Within-participant differences in gait variability: '
        'high minus low environmental exposure.'
    )
)


In [7]:
Path('results').mkdir(exist_ok=True)
table4.to_csv('results/table4.csv',index=False)

In [8]:

from pathlib import Path
import numpy as np
import pandas as pd
from scipy.stats import wilcoxon
from statsmodels.stats.multitest import multipletests

BOUTS_PATH = Path("derived/bouts.parquet")
RESULTS_DIR = Path("results")
RESULTS_DIR.mkdir(exist_ok=True)

MIN_BOUTS_PER_PARTICIPANT = 10
STEP_THRESHOLDS = (5, 10)
N_BOOT = 5000
BOOTSTRAP_SEED = 7

OUTCOMES = {
    "STCoV": "BoutStepTCV",
    "SaTCoV": "BoutStanceTCV",
    "SwTCoV": "BoutSwingTCV",
    "SrTCoV": "BoutStrideTCV",
}

EXPOSURES = {
    "Open Floor High vs Low": "EnvVisibleOpenFloorRatioMean",
    "Clutter High vs Low": "EnvLowerCentralObstacleAreaRatioMean",
    "Corridor Occupancy High vs Low": "EnvCentralCorridorOccupancyMean",
    "Path Obstruction High vs Low": "EnvPathObstructionIndexMean",
    "Stairs High vs Low": "EnvStairsPresentFraction",
}

bouts = pd.read_parquet(BOUTS_PATH)

missing = (
    set(OUTCOMES.values())
    | set(EXPOSURES.values())
    | {"Participant", "BoutNumSteps"}
) - set(bouts.columns)

if missing:
    raise ValueError(f"Missing required columns: {sorted(missing)}")

bouts = bouts.loc[
    ~bouts["Participant"].str.startswith("TOPAL", na=False)
].copy()


bouts["Group"] = np.where(
    bouts["Participant"].str.contains("PD", regex=False),
    "PwP",
    "Control",
)


In [9]:

def participant_high_low_deltas(df, exposure_col, outcome_col):
    complete = df[
        ["Participant", exposure_col, outcome_col]
    ].replace([np.inf, -np.inf], np.nan).dropna()

    deltas = []
    bout_counts = []
    low_counts = []
    high_counts = []

    for participant, participant_bouts in complete.groupby("Participant"):
        n = len(participant_bouts)

        if n < MIN_BOUTS_PER_PARTICIPANT:
            continue

        threshold = participant_bouts[exposure_col].median()

        low = participant_bouts.loc[
            participant_bouts[exposure_col] <= threshold,
            outcome_col,
        ]

        high = participant_bouts.loc[
            participant_bouts[exposure_col] > threshold,
            outcome_col,
        ]

        if low.empty or high.empty:
            continue

        deltas.append(float(high.mean() - low.mean()))
        bout_counts.append(n)
        low_counts.append(len(low))
        high_counts.append(len(high))

    return {
        "deltas": np.asarray(deltas, dtype=float),
        "n_participants": len(deltas),
        "n_bouts": sum(bout_counts),
        "median_bouts": (
            float(np.median(bout_counts)) if bout_counts else np.nan
        ),
        "median_low_bouts": (
            float(np.median(low_counts)) if low_counts else np.nan
        ),
        "median_high_bouts": (
            float(np.median(high_counts)) if high_counts else np.nan
        ),
    }


In [10]:

all_tables = {}

for min_steps in STEP_THRESHOLDS:

    eligible = bouts.loc[
        bouts["BoutNumSteps"] >= min_steps
    ].copy()

    rows = []

    for exposure_label, exposure_col in EXPOSURES.items():
        for outcome_label, outcome_col in OUTCOMES.items():
            for group in ["PwP", "Control"]:

                result = participant_high_low_deltas(
                    eligible.loc[eligible["Group"] == group],
                    exposure_col,
                    outcome_col,
                )

                values = result["deltas"]

                stats = (
                    summarize_deltas(values)
                    if len(values) > 0
                    else {
                        "Mean Δ": np.nan,
                        "Median Δ": np.nan,
                        "95% CI lower": np.nan,
                        "95% CI upper": np.nan,
                        "Fraction positive": np.nan,
                        "Wilcoxon p": np.nan,
                    }
                )

                rows.append({
                    "Environmental factor": exposure_label,
                    "Group": group,
                    "Outcome": outcome_label,
                    "Minimum steps": min_steps,
                    "Participants (n)": result["n_participants"],
                    "Bouts (n)": result["n_bouts"],
                    "Median bouts/participant": result["median_bouts"],
                    "Median low bouts": result["median_low_bouts"],
                    "Median high bouts": result["median_high_bouts"],
                    **stats,
                })

    table = pd.DataFrame(rows)

    valid = table["Wilcoxon p"].notna()

    table["Holm p"] = np.nan
    table.loc[valid, "Holm p"] = multipletests(
        table.loc[valid, "Wilcoxon p"],
        method="holm",
    )[1]

    table["Holm significant"] = table["Holm p"] < 0.05

    all_tables[min_steps] = table

    table.to_csv(
        RESULTS_DIR / f"table4_min{min_steps}_steps.csv",
        index=False,
    )

table4_5 = all_tables[5]
table4_10 = all_tables[10]

display(table4_5)
display(table4_10)


,Environmental factor,Group,Outcome,Minimum steps,Participants (n),Bouts (n),Median bouts/participant,Median low bouts,Median high bouts,Mean Δ,Median Δ,95% CI lower,95% CI upper,Fraction positive,Wilcoxon p,Holm p,Holm significant
0,Open Floor High vs Low,PwP,STCoV,5,18,1336,61.5,31.0,30.5,-0.027499,-0.016854,-0.045472,-0.011094,0.222222,0.002808,0.098267,False
1,Open Floor High vs Low,Control,STCoV,5,17,1164,72.0,36.0,36.0,-0.028987,-0.029970,-0.041785,-0.017278,0.058824,0.000107,0.004166,True
2,Open Floor High vs Low,PwP,SaTCoV,5,18,1336,61.5,31.0,30.5,-0.025341,-0.024973,-0.038764,-0.011795,0.277778,0.005600,0.167999,False
3,Open Floor High vs Low,Control,SaTCoV,5,17,1164,72.0,36.0,36.0,-0.029589,-0.025203,-0.042102,-0.019187,0.058824,0.000031,0.001221,True
4,Open Floor High vs Low,PwP,SwTCoV,5,18,1336,61.5,31.0,30.5,-0.016469,-0.012330,-0.029920,-0.003417,0.333333,0.043167,0.863342,False
5,Open Floor High vs Low,Control,SwTCoV,5,17,1164,72.0,36.0,36.0,-0.017760,-0.017697,-0.027625,-0.008794,0.235294,0.001343,0.051025,False
6,Open Floor High vs Low,PwP,SrTCoV,5,18,1336,61.5,31.0,30.5,-0.014336,-0.010524,-0.025423,-0.004085,0.222222,0.023674,0.580597,False
7,Open Floor High vs Low,Control,SrTCoV,5,17,1164,72.0,36.0,36.0,-0.013187,-0.009001,-0.023177,-0.005212,0.235294,0.004639,0.143799,False
8,Clutter High vs Low,PwP,STCoV,5,18,1336,61.5,31.0,30.5,0.024401,0.028816,0.010740,0.036987,0.833333,0.003365,0.107666,False
9,Clutter High vs Low,Control,STCoV,5,17,1164,72.0,36.0,36.0,0.014045,0.010566,0.003998,0.024559,0.764706,0.017426,0.453064,False


,Environmental factor,Group,Outcome,Minimum steps,Participants (n),Bouts (n),Median bouts/participant,Median low bouts,Median high bouts,Mean Δ,Median Δ,95% CI lower,95% CI upper,Fraction positive,Wilcoxon p,Holm p,Holm significant
0,Open Floor High vs Low,PwP,STCoV,10,18,736,33.0,16.5,16.5,-0.013990,-0.010341,-0.024524,-0.004078,0.277778,0.030365,0.941315,False
1,Open Floor High vs Low,Control,STCoV,10,16,580,32.5,16.5,16.0,-0.021873,-0.025615,-0.036837,-0.006983,0.312500,0.018250,0.675232,False
2,Open Floor High vs Low,PwP,SaTCoV,10,18,736,33.0,16.5,16.5,-0.013218,-0.008749,-0.023735,-0.003298,0.277778,0.038490,1.000000,False
3,Open Floor High vs Low,Control,SaTCoV,10,16,580,32.5,16.5,16.0,-0.017631,-0.017212,-0.031825,-0.004603,0.250000,0.024963,0.848755,False
4,Open Floor High vs Low,PwP,SwTCoV,10,18,736,33.0,16.5,16.5,-0.010162,-0.008997,-0.018964,-0.001173,0.277778,0.026848,0.859131,False
5,Open Floor High vs Low,Control,SwTCoV,10,16,580,32.5,16.5,16.0,-0.015548,-0.015465,-0.028973,-0.002140,0.312500,0.093445,1.000000,False
6,Open Floor High vs Low,PwP,SrTCoV,10,18,736,33.0,16.5,16.5,-0.006670,-0.004177,-0.014433,0.000245,0.222222,0.043167,1.000000,False
7,Open Floor High vs Low,Control,SrTCoV,10,16,580,32.5,16.5,16.0,-0.010778,-0.012223,-0.021636,-0.000007,0.375000,0.129730,1.000000,False
8,Clutter High vs Low,PwP,STCoV,10,18,736,33.0,16.5,16.5,0.019247,0.021239,-0.000583,0.036030,0.777778,0.015930,0.621277,False
9,Clutter High vs Low,Control,STCoV,10,16,580,32.5,16.5,16.0,0.021876,0.022864,0.006121,0.040501,0.750000,0.038635,1.000000,False


In [11]:
for group in ["PwP", "Control"]:
    participants = sorted(
        bouts.loc[bouts["Group"] == group, "Participant"].unique()
    )
    print(f"\n{group}: {len(participants)}")
    print("\n".join(participants))


PwP: 19
UFALL_PD_0001
UFALL_PD_0002
UFALL_PD_0003
UFALL_PD_0004
UFALL_PD_0005
UFALL_PD_0007
UFALL_PD_0008
UFALL_PD_0009
UFALL_PD_0010
UFALL_PD_0011
UFALL_PD_0012
UFALL_PD_0014
UFALL_PD_0015
UFALL_PD_0016
UFALL_PD_0017
UFALL_PD_0019
UFALL_PD_0020
UFALL_PD_0021
UFALL_PD_0022

Control: 18
UFALL_CL_0001
UFALL_CL_0002
UFALL_CL_0003
UFALL_CL_0005
UFALL_CL_0008
UFALL_CL_0009
UFALL_CL_0010
UFALL_CL_0011
UFALL_CL_0012
UFALL_CL_0013
UFALL_CL_0014
UFALL_CL_0015
UFALL_CL_0016
UFALL_CL_0017
UFALL_CL_0018
UFALL_CL_0019
UFALL_CL_0020
UFALL_CL_0021
